# Entrenar el clasificador de notas OSM (Colab, GPU)

Entrena la línea base (TF-IDF) y SetFit para **categoría** y **tema** con las
etiquetas del labeler, y muestra las estadísticas.

1. Menú **Entorno de ejecución → Cambiar tipo → GPU (T4)**.
2. Corre las celdas en orden.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código y librerías

In [ ]:
!git clone -q https://github.com/Rub21/osm-notes-deploy.git /content/osm-notes-deploy || git -C /content/osm-notes-deploy pull -q
!pip install -q setfit==1.1.0 sentence-transformers==3.3.1 transformers==4.45.2 datasets==3.1.0

## 2. Etiquetas

Se bajan de la API pública del labeler. Filtros opcionales en `FILTERS`:
`lang=es,en`, `user=Rub21`, `since=2026-10-30`, `category=spam,personal_data`,
`theme=food`, `changed=1` (solo donde la persona corrigió a la IA).

In [ ]:
import urllib.request
import pandas as pd

FILTERS = ""  # ejemplo: "lang=es,en&since=2026-10-30"
URL = "https://osm-notes.rub21.com/api/labels.csv" + (f"?{FILTERS}" if FILTERS else "")
urllib.request.urlretrieve(URL, "/content/labels.csv")

labels = pd.read_csv("/content/labels.csv", encoding="utf-8-sig")
print(len(labels), "etiquetas")
labels[["lang", "category", "theme"]].describe()

## 2b. Estadísticas de los datos

Antes de entrenar: cuántas etiquetas hay por clase, idioma y plantilla, y
cuánto acierta la IA. Una clase con menos de 8 etiquetas queda fuera del
entrenamiento; con menos de ~20 el modelo la aprende mal.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

df = labels.copy()
df["template"] = df["template"].fillna("(sin plantilla)")
df["user_text"] = df["user_text"].fillna("")
df["words"] = df["user_text"].str.split().str.len()
df["updated_at"] = pd.to_datetime(df["updated_at"], utc=True, errors="coerce")

print(f"Etiquetas: {len(df)} · notas: {df.note_id.nunique()} · personas: {df.user_name.nunique()}")
print(f"Notas con más de una etiqueta: {(df.note_id.value_counts() > 1).sum()}")

fig, ax = plt.subplots(2, 3, figsize=(20, 10))
def bars(s, a, title, min_line=None):
    s.sort_values().plot.barh(ax=a, title=title)
    for i, v in enumerate(s.sort_values()):
        a.text(v, i, f" {v}", va="center", fontsize=8)
    if min_line:
        a.axvline(min_line, color="red", ls="--", lw=1)
bars(df["category"].value_counts(), ax[0, 0], "Por categoría (rojo = 20)", 20)
bars(df["theme"].fillna("(vacío)").value_counts(), ax[0, 1], "Por tema (rojo = 20)", 20)
bars(df["lang"].value_counts().head(15), ax[0, 2], "Por idioma (top 15)")
bars(df["template"].value_counts(), ax[1, 0], "Por plantilla")
bars(df["user_name"].value_counts().head(15), ax[1, 1], "Por persona")
df["updated_at"].dt.strftime("%Y-%m-%d").value_counts().sort_index().plot.bar(ax=ax[1, 2], title="Etiquetas por día", rot=45)
plt.tight_layout(); plt.show()

In [ ]:
# Categoría x tema: qué temas aparecen en cada categoría.
ct = pd.crosstab(df["category"], df["theme"].fillna("(vacío)"))
fig, a = plt.subplots(figsize=(14, 6))
a.imshow(ct.values, cmap="Greens")
a.set_xticks(range(len(ct.columns)), ct.columns, rotation=45, ha="right")
a.set_yticks(range(len(ct.index)), ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        if ct.values[i, j]:
            a.text(j, i, ct.values[i, j], ha="center", va="center", fontsize=8)
a.set_title("Categoría x tema"); plt.tight_layout(); plt.show()

# Largo del texto de la persona (palabras), por categoría.
display(df.groupby("category")["words"].describe()[["count", "mean", "50%", "max"]].round(1))

### ¿Cuánto acierta la IA?

Compara `claude_category` / `claude_theme` con la etiqueta final. Si da 100%,
las páginas se aprobaron sin revisar y el modelo solo aprenderá a copiar a la IA.

In [ ]:
ai = df[df["claude_category"].notna()]
if len(ai):
    ok_cat = (ai["category"] == ai["claude_category"]).mean()
    ok_th = (ai["theme"] == ai["claude_theme"]).mean()
    print(f"Con sugerencia de IA: {len(ai)} · acierto categoría {ok_cat:.1%} · tema {ok_th:.1%}")
    per = ai.assign(ok=ai["category"] == ai["claude_category"]).groupby("claude_category")["ok"] \
            .agg(["count", "mean"]).rename(columns={"count": "n", "mean": "acierto"}).sort_values("acierto")
    display(per.style.format({"acierto": "{:.0%}"}))
    changed = ai[ai["category"] != ai["claude_category"]]
    if len(changed):
        print("Qué corrigió la persona (fila = IA, columna = persona):")
        display(pd.crosstab(changed["claude_category"], changed["category"]))
else:
    print("No hay sugerencias de IA en estas etiquetas.")

## 3. Entrenar

`BASE_MODEL`: `paraphrase-multilingual-mpnet-base-v2` (mejor) o
`paraphrase-multilingual-MiniLM-L12-v2` (más rápido). En una T4 son unos minutos.

In [ ]:
BASE_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
ARGS = f"--base-model {BASE_MODEL} --train-langs es,en,und --test-size 0.2 --min-per-class 8 --iterations 20 --epochs 1 --batch-size 32 --seed 42"
OUT = "/content/output"
TRAIN = "/content/osm-notes-deploy/notebooks/train.py"

!python {TRAIN} --from-csv /content/labels.csv --output {OUT} --target category {ARGS}
!python {TRAIN} --from-csv /content/labels.csv --output {OUT} --target theme {ARGS}

## 4. Estadísticas

In [ ]:
import glob, json
import matplotlib.pyplot as plt

def last_run(target):
    return sorted(glob.glob(f"{OUT}/run_*_{target}"))[-1]

def show(target):
    run = last_run(target)
    m = json.load(open(f"{run}/metrics.json"))
    pred = pd.read_csv(f"{run}/predictions.csv")
    best = "pred_setfit" if "pred_setfit" in pred else "pred_baseline"
    test = pred[pred["split"] == "test"]
    print(f"== {target}  ({run})  train={m['train']} test={m['test']}")
    display(pd.DataFrame(m["summary"]).T)

    fig, ax = plt.subplots(1, 3, figsize=(20, 5))
    s = pd.DataFrame(m["summary"]).T["macro_f1"]
    s.plot.bar(ax=ax[0], title="macro-F1 por modelo", ylim=(0, 1), rot=0)

    pc = pd.DataFrame(m["per_class"]).set_index("class")
    pc[["precision", "recall", "f1"]].plot.barh(ax=ax[1], title=f"Por clase ({best[5:]})", xlim=(0, 1))

    cm = pd.crosstab(test["label"], test[best])
    im = ax[2].imshow(cm.values, cmap="Blues")
    ax[2].set_xticks(range(len(cm.columns)), cm.columns, rotation=60, ha="right")
    ax[2].set_yticks(range(len(cm.index)), cm.index)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax[2].text(j, i, cm.values[i, j], ha="center", va="center", fontsize=8)
    ax[2].set_title("Matriz de confusión (filas = real)")
    plt.tight_layout(); plt.show()

    display(pd.DataFrame(m["per_lang"]))
    return run

run_cat = show("category")
run_theme = show("theme")

### Línea base vs SetFit por clase

Si SetFit no le gana a la línea base en una clase, faltan ejemplos de esa clase
o las etiquetas son inconsistentes.

In [ ]:
from sklearn.metrics import f1_score

def compare(run):
    pred = pd.read_csv(f"{run}/predictions.csv")
    t = pred[pred["split"] == "test"]
    cols = [c for c in ("pred_baseline", "pred_setfit") if c in t]
    classes = sorted(t["label"].unique())
    out = pd.DataFrame({c[5:]: f1_score(t["label"], t[c], labels=classes, average=None, zero_division=0)
                        for c in cols}, index=classes)
    out["support"] = t["label"].value_counts().reindex(classes).values
    if "setfit" in out:
        out["gana"] = (out["setfit"] - out["baseline"]).round(2)
    ax = out[[c[5:] for c in cols]].plot.barh(figsize=(10, 6), xlim=(0, 1), title=f"F1 por clase · {run.split('/')[-1]}")
    plt.tight_layout(); plt.show()
    return out.round(2)

display(compare(run_cat))
display(compare(run_theme))

### ¿Qué tan seguro está el modelo?

Acierto por largo del texto: las notas muy cortas suelen fallar más.

In [ ]:
def by_length(run):
    pred = pd.read_csv(f"{run}/predictions.csv")
    t = pred[pred["split"] == "test"].copy()
    col = "pred_setfit" if "pred_setfit" in t else "pred_baseline"
    t["words"] = t["user_text"].fillna("").str.split().str.len()
    t["largo"] = pd.cut(t["words"], [-1, 0, 3, 10, 30, 10_000], labels=["0", "1-3", "4-10", "11-30", ">30"])
    return t.assign(ok=t["label"] == t[col]).groupby("largo", observed=False)["ok"] \
            .agg(["count", "mean"]).rename(columns={"count": "n", "mean": "acierto"}).round(2)

print("Categoría"); display(by_length(run_cat))
print("Tema"); display(by_length(run_theme))

## 5. Errores (notas mal clasificadas)

In [ ]:
err = pd.read_csv(f"{run_cat}/errors.csv")
pd.set_option("display.max_colwidth", 200)
err.head(50)

## 6. Descargar el modelo

In [ ]:
from google.colab import files
!cd {OUT} && zip -qr /content/models.zip .
files.download("/content/models.zip")